# Entregable 1 — Prueba de falla del modelo pequeño
### IA Generativa (580694) · Proyecto semestral · Facultad de Ingeniería UdeC

**Qué hace este cuaderno**

1. Carga un modelo abierto y pequeño (**Qwen3-4B**, nuestro modelo principal) en la GPU gratis de Colab → **evidencia de viabilidad de ejecución**.
2. Le hace **10 preguntas** sobre la normativa de pregrado de Ingeniería **sin darle los documentos** (prompting directo).
3. Arma la **tabla de resultados** que muestra al modelo inventando artículos y cifras → **evidencia del diagnóstico de la falla**.

Las 10 preguntas están repartidas en **5 categorías**: factual, numérica, condicional, cruce entre documentos y abstención.

> **IMPORTANTE — corre las celdas de arriba hacia abajo, en orden.** Si Colab se reinicia (o sale un `NameError`), simplemente vuelve a correr desde el **Paso 3**.
>
> Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno → GPU (T4)**.

---

## Paso 1 — Comprobar la GPU

In [1]:
import torch
print("¿GPU disponible?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Memoria total (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

¿GPU disponible?: True
GPU: Tesla T4
Memoria total (GB): 15.6


## Paso 2 — Instalar la librería
Qwen3 necesita `transformers` reciente (≥ 4.51).

In [2]:
!pip install -q -U "transformers>=4.51.0" accelerate
print("Instalación lista.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 85.9 MB/s eta 0:00:00
Instalación lista.


## Paso 3 — Cargar el modelo pequeño (Qwen3-4B)
~4 mil millones de parámetros, licencia **Apache 2.0**, 100+ idiomas. La primera vez descarga el modelo (unos minutos).

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen3-4B"   # modelo principal (para el bono: notablemente < 8B)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto", device_map="auto")
print("Modelo cargado:", MODEL_NAME)
print("Memoria GPU usada (GB):", round(torch.cuda.memory_allocated()/1e9, 2))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo cargado: Qwen/Qwen3-4B
Memoria GPU usada (GB): 8.05


## Paso 4 — Función para preguntar
Fijamos el rol del modelo y `preguntar()`. Puntos clave para que la prueba sea justa:
- Le pedimos **citar el artículo** y **decir si no sabe** (si igual inventa, la culpa es del modelo, no del prompt).
- Modo **sin "thinking"** y decodificación **determinista** (`do_sample=False`): reproducible.
- **No le damos los documentos**: esto es el baseline que queremos que falle.

In [4]:
SYSTEM = (
    "Eres un asistente experto en la normativa de pregrado de la Facultad de "
    "Ingeniería de la Universidad de Concepción. Responde de forma breve, con el "
    "dato exacto y citando el artículo correspondiente (por ejemplo: 'Art. 8'). "
    "Si no tienes la información, di explícitamente que no está en la normativa."
)

def preguntar(pregunta, sistema=SYSTEM, max_new_tokens=256):
    messages = [{"role":"system","content":sistema}, {"role":"user","content":pregunta}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

print("Prueba rápida:")
print(preguntar("¿Cuál es la nota mínima para aprobar una asignatura en Ingeniería?"))

Prueba rápida:
La nota mínima para aprobar una asignatura en Ingeniería es **4.0** (Art. 8).


## Paso 5 — El test set (10 preguntas verificadas)
Cada pregunta tiene su respuesta correcta (**gold**) y la fuente, sacadas directo de los documentos oficiales.

- **RG** = Reglamento General de Docencia de Pregrado · **RI-FI** = Reglamento Interno de Docencia de Pregrado, Facultad de Ingeniería · **CAL** = Calendario de Docencia de Pregrado 2026

**Sobre la categoría "abstención":** son preguntas cuya respuesta **no está regulada en ningún documento** (la eximición la fija cada asignatura; el tamaño de sección no está normado). La respuesta correcta es **reconocer que no hay un dato fijo / no está en la normativa**. Si el modelo inventa un número, falla.

In [5]:
preguntas = [
    {"categoria":"factual", "pregunta":"¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?",
     "gold_dato":"4,0", "gold_fuente":"Art. 11, RI-FI"},
    {"categoria":"factual", "pregunta":"¿Cuántas evaluaciones sumativas como mínimo debe tener una asignatura?",
     "gold_dato":"al menos 3 (tres)", "gold_fuente":"Art. 11, RI-FI"},

    {"categoria":"numerica", "pregunta":"¿Cuál es el mínimo de créditos que debo inscribir por período en Ingeniería?",
     "gold_dato":"8 créditos (igual o superior a 8)", "gold_fuente":"Art. 8, RI-FI"},
    {"categoria":"numerica", "pregunta":"¿Con menos de cuántos créditos aprobados al término del segundo semestre quedo en causal de baja académica?",
     "gold_dato":"menos de 15 créditos", "gold_fuente":"Art. 14 a), RI-FI"},

    {"categoria":"condicional", "pregunta":"Si vengo de otra universidad y quiero ingresar a Ingeniería, ¿qué promedio mínimo necesito? ¿Y si en cambio me cambio entre carreras dentro de Ingeniería?",
     "gold_dato":"4,5 desde otra universidad/facultad; 4,2 para cambio dentro de Ingeniería", "gold_fuente":"Art. 17 y Art. 18, RI-FI"},
    {"categoria":"condicional", "pregunta":"¿Qué promedio mínimo se exige para cursar una segunda carrera de forma simultánea?",
     "gold_dato":"5,0", "gold_fuente":"Art. 19, RI-FI"},

    {"categoria":"cruce", "pregunta":"Quiero modificar las asignaturas que inscribí este segundo semestre 2026. ¿Hasta qué fecha puedo hacerlo?",
     "gold_dato":"hasta el 04 de septiembre de 2026", "gold_fuente":"CAL 2do sem 2026 (la fecha está en el Calendario; la figura, en el RG/RI-FI)"},
    {"categoria":"cruce", "pregunta":"Si el Reglamento Interno de Ingeniería contradice al Reglamento General de Docencia, ¿cuál prevalece?",
     "gold_dato":"prevalece el Reglamento General (y las Normas de Ingreso)", "gold_fuente":"Art. 34, RI-FI"},

    {"categoria":"abstencion", "pregunta":"¿Con qué nota mínima puedo eximirme del examen final de una asignatura en Ingeniería?",
     "gold_dato":"[no regulado en el corpus -> debe abstenerse; lo fija cada asignatura]", "gold_fuente":"no está en RG/RI-FI/CAL"},
    {"categoria":"abstencion", "pregunta":"¿Cuál es el número máximo de alumnos por sección en una asignatura de Ingeniería?",
     "gold_dato":"[no regulado en el corpus -> debe abstenerse]", "gold_fuente":"no está en RG/RI-FI/CAL"},
]
print("Total de preguntas:", len(preguntas))

Total de preguntas: 10


## Paso 6 — Correr las 10 preguntas y armar la tabla
Genera la tabla que vas a **capturar para el póster**. La columna `correcto` queda vacía; la llenas en el Paso 7.

In [6]:
import pandas as pd

filas = []
for p in preguntas:
    resp = preguntar(p["pregunta"])
    filas.append({
        "categoria": p["categoria"],
        "pregunta": p["pregunta"],
        "respuesta_modelo": resp,
        "gold_dato": p["gold_dato"],
        "gold_fuente": p["gold_fuente"],
        "correcto": "",   # se llena en el Paso 7
    })
    print("="*90)
    print("[", p["categoria"], "]", p["pregunta"])
    print("-> MODELO:", resp)
    print("-> GOLD  :", p["gold_dato"], "(", p["gold_fuente"], ")")

df = pd.DataFrame(filas)
df.to_csv("resultados_qwen3_4b.csv", index=False)
print("\n\nTabla guardada en resultados_qwen3_4b.csv")
df

[ factual ] ¿Cuál es la nota mínima para aprobar una asignatura en la Facultad de Ingeniería?
-> MODELO: La nota mínima para aprobar una asignatura en la Facultad de Ingeniería de la Universidad de Concepción es **4.0** (Art. 8).
-> GOLD  : 4,0 ( Art. 11, RI-FI )
[ factual ] ¿Cuántas evaluaciones sumativas como mínimo debe tener una asignatura?
-> MODELO: La normativa de pregrado de la Facultad de Ingeniería de la Universidad de Concepción no especifica un número mínimo de evaluaciones sumativas que debe tener una asignatura. Por lo tanto, esta información no está en la normativa.
-> GOLD  : al menos 3 (tres) ( Art. 11, RI-FI )
[ numerica ] ¿Cuál es el mínimo de créditos que debo inscribir por período en Ingeniería?
-> MODELO: El mínimo de créditos que debe inscribirse por período en Ingeniería es **60 créditos** (Art. 8).
-> GOLD  : 8 créditos (igual o superior a 8) ( Art. 8, RI-FI )
[ numerica ] ¿Con menos de cuántos créditos aprobados al término del segundo semestre quedo en causal 

,categoria,pregunta,respuesta_modelo,gold_dato,gold_fuente,correcto
0,factual,¿Cuál es la nota mínima para aprobar una asign...,La nota mínima para aprobar una asignatura en ...,"4,0","Art. 11, RI-FI",
1,factual,¿Cuántas evaluaciones sumativas como mínimo de...,La normativa de pregrado de la Facultad de Ing...,al menos 3 (tres),"Art. 11, RI-FI",
2,numerica,¿Cuál es el mínimo de créditos que debo inscri...,El mínimo de créditos que debe inscribirse por...,8 créditos (igual o superior a 8),"Art. 8, RI-FI",
3,numerica,¿Con menos de cuántos créditos aprobados al té...,Art. 12. Queda en causal de baja académica el ...,menos de 15 créditos,"Art. 14 a), RI-FI",
4,condicional,Si vengo de otra universidad y quiero ingresar...,Para ingresar a la carrera de Ingeniería en la...,"4,5 desde otra universidad/facultad; 4,2 para ...","Art. 17 y Art. 18, RI-FI",
5,condicional,¿Qué promedio mínimo se exige para cursar una ...,La normativa de la Facultad de Ingeniería de l...,"5,0","Art. 19, RI-FI",
6,cruce,Quiero modificar las asignaturas que inscribí ...,La fecha límite para realizar modificaciones e...,hasta el 04 de septiembre de 2026,CAL 2do sem 2026 (la fecha está en el Calendar...,
7,cruce,Si el Reglamento Interno de Ingeniería contrad...,Según el Art. 10 del Reglamento Interno de Ing...,prevalece el Reglamento General (y las Normas ...,"Art. 34, RI-FI",
8,abstencion,¿Con qué nota mínima puedo eximirme del examen...,La normativa de pregrado de la Facultad de Ing...,[no regulado en el corpus -> debe abstenerse; ...,no está en RG/RI-FI/CAL,
9,abstencion,¿Cuál es el número máximo de alumnos por secci...,No está especificado en la normativa de pregra...,[no regulado en el corpus -> debe abstenerse],no está en RG/RI-FI/CAL,


## Paso 7 — Calificar y resumir
Llena la lista `veredictos` con `"si"` / `"no"` para las 10 preguntas, **en orden (0 a 9)**.

**Regla estricta:**
- Categorías con respuesta en el corpus (factual, numérica, condicional, cruce): `"si"` solo si acierta **el dato Y el artículo**.
- Abstención: `"si"` solo si **reconoce que no está regulado / no da un número**; si inventa una cifra, `"no"`.

> Si sale `NameError: df`, es que el entorno se reinició → vuelve a correr desde el Paso 3.

In [9]:
veredictos = ["no", "no", "no", "no", "no", "no", "no", "no", "si", "si"]

assert len(veredictos) == len(df), "Deben ser 10 veredictos."
df["correcto"] = veredictos
df.to_csv("resultados_qwen3_4b.csv", index=False)

df["acierto"] = df["correcto"].str.strip().str.lower().isin(["si","sí"])
resumen = df.groupby("categoria")["acierto"].agg(["sum","count"])
resumen["exactitud_%"] = (100*resumen["sum"]/resumen["count"]).round(0)
print("EXACTITUD POR CATEGORÍA (", MODEL_NAME, ")")
print(resumen)
print("\nExactitud global: {:.0f}%".format(100*df["acierto"].mean()))
df[["categoria","pregunta","respuesta_modelo","gold_dato","correcto"]]

EXACTITUD POR CATEGORÍA ( Qwen/Qwen3-4B )
             sum  count  exactitud_%
categoria                           
abstencion     2      2        100.0
condicional    0      2          0.0
cruce          0      2          0.0
factual        0      2          0.0
numerica       0      2          0.0

Exactitud global: 20%


,categoria,pregunta,respuesta_modelo,gold_dato,correcto
0,factual,¿Cuál es la nota mínima para aprobar una asign...,La nota mínima para aprobar una asignatura en ...,"4,0",no
1,factual,¿Cuántas evaluaciones sumativas como mínimo de...,La normativa de pregrado de la Facultad de Ing...,al menos 3 (tres),no
2,numerica,¿Cuál es el mínimo de créditos que debo inscri...,El mínimo de créditos que debe inscribirse por...,8 créditos (igual o superior a 8),no
3,numerica,¿Con menos de cuántos créditos aprobados al té...,Art. 12. Queda en causal de baja académica el ...,menos de 15 créditos,no
4,condicional,Si vengo de otra universidad y quiero ingresar...,Para ingresar a la carrera de Ingeniería en la...,"4,5 desde otra universidad/facultad; 4,2 para ...",no
5,condicional,¿Qué promedio mínimo se exige para cursar una ...,La normativa de la Facultad de Ingeniería de l...,"5,0",no
6,cruce,Quiero modificar las asignaturas que inscribí ...,La fecha límite para realizar modificaciones e...,hasta el 04 de septiembre de 2026,no
7,cruce,Si el Reglamento Interno de Ingeniería contrad...,Según el Art. 10 del Reglamento Interno de Ing...,prevalece el Reglamento General (y las Normas ...,no
8,abstencion,¿Con qué nota mínima puedo eximirme del examen...,La normativa de pregrado de la Facultad de Ing...,[no regulado en el corpus -> debe abstenerse; ...,si
9,abstencion,¿Cuál es el número máximo de alumnos por secci...,No está especificado en la normativa de pregra...,[no regulado en el corpus -> debe abstenerse],si


## Paso 8 (opcional) — "El grande también falla"
Corre las mismas preguntas en **Qwen3-8B** (el tope permitido). Si el grande también inventa, demuestras que la falla es **estructural** (no por elegir un modelo chico) → blinda el diagnóstico y el bono.

> Reinicia el entorno antes (Runtime → Restart) para no quedarte sin memoria, luego corre Pasos 2, 4, 5, 6 con esta línea cambiada:
```python
MODEL_NAME = "Qwen/Qwen3-8B"
```

---
### Cómo usar esto en el Entregable 1
- **Viabilidad (10 pts):** captura del Paso 1 + Paso 3 (GPU + modelo cargado).
- **Diagnóstico (10 pts):** captura de la tabla del Paso 6 y del resumen del Paso 7.
- Descarga `resultados_qwen3_4b.csv` desde el panel de archivos (carpeta 📁 a la izquierda).
- **Siguiente:** ampliar el test set a ~50 preguntas (10 por categoría) con `preguntas_semilla.csv`.